# t-stack-trajectory — worked example 1: Stack trajectory with time on axis 0

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `t-stack-trajectory`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

`torch.stack(tensors, dim=d)` inserts a new axis at position `d` and fills it with the stacked elements. When you have a list of T tensors each of shape `(B, D)` and stack with `dim=0`, the result is `(T, B, D)` — time first, then batch. This is the convention used when you want to iterate over timesteps in the outer loop.

## Worked solution

**Step 1 — Understand what stack does.**
Unlike `torch.cat`, which concatenates along an existing axis, `torch.stack` creates a new axis. All tensors in the list must have identical shapes.

**Step 2 — Choose dim=0 for time-first.**
If each element in the list is a `(B, D)` snapshot, stacking with `dim=0` puts the list index as the first axis: the result has shape `(T, B, D)`. Slicing `result[t]` gives the batch snapshot at step `t`.

**Step 3 — Contrast with dim=1.**
Stacking with `dim=1` gives `(B, T, D)` — batch-first, time second. The choice depends on how downstream code accesses the trajectory.

**Step 4 — Simulate a simple optimization trajectory.**
At each gradient-descent step, we clone the current parameter vector and append it to a list. After the loop, we stack into a single tensor for analysis or plotting.

In [ ]:
import torch as t

t.manual_seed(42)
B, D, T = 4, 3, 6

# Simulate T gradient-descent steps for B parameter vectors
params = t.randn(B, D)  # initial parameters
step_snapshots = []
for step in range(T):
    step_snapshots.append(params.clone())
    params = params - 0.1 * t.randn(B, D)  # fake gradient update

# Stack with dim=0 -> (T, B, D)
traj = t.stack(step_snapshots, dim=0)
print('Number of snapshots:', len(step_snapshots))  # 6
print('Each snapshot shape:', step_snapshots[0].shape)  # (4, 3)
print('Stacked trajectory shape:', traj.shape)         # (6, 4, 3)
print('Step 0, batch 0:', traj[0, 0])   # first snapshot, first item
print('Step 5, batch 0:', traj[5, 0])   # last snapshot, first item